In [17]:
import numpy as np
import pandas as pd

# Generating data

y = (x * z)³

In [18]:
x_data = np.random.uniform(low=0, high=1.0, size=(3000,))
#z_data = np.random.standard_normal(size=(3000,))
z_data = np.random.uniform(low=0, high=1.0, size=(3000,))
pow_data = np.power(x_data * z_data, 1) 
multi_data = x_data * z_data

df_power = pd.DataFrame(np.array([x_data, z_data, pow_data]).T, columns=['x_data', 'z_data', 'y_data'])
df_multi = pd.DataFrame(np.array([x_data, z_data, multi_data]).T, columns=['x_data', 'z_data', 'y_data'])
df_power

,x_data,z_data,y_data
0,0.690434,0.244386,0.168733
1,0.841309,0.830479,0.698689
2,0.193507,0.976935,0.189044
3,0.102527,0.169020,0.017329
4,0.445879,0.882088,0.393305
...,...,...,...
2995,0.351978,0.382016,0.134461
2996,0.545153,0.690897,0.376645
2997,0.494676,0.148344,0.073382
2998,0.778529,0.634569,0.494030


In [19]:
df_multi

,x_data,z_data,y_data
0,0.690434,0.244386,0.168733
1,0.841309,0.830479,0.698689
2,0.193507,0.976935,0.189044
3,0.102527,0.169020,0.017329
4,0.445879,0.882088,0.393305
...,...,...,...
2995,0.351978,0.382016,0.134461
2996,0.545153,0.690897,0.376645
2997,0.494676,0.148344,0.073382
2998,0.778529,0.634569,0.494030


In [20]:
from spice_net import spice_net_som

spice_net_som.local_min(-4, 4, 0.05, 0.005, lambda x: x ** 2)

8.599438671404554e-06

In [21]:
import spice_net as spn
som_size = 100

som_1 = spn.SpiceNetSom(n_neurons=som_size,
                        value_range_start=-1,
                        value_range_end=1,
                        lrf_tuning_curve=spn.ConstLRF(0.8),
                        lrf_interaction_kernel=spn.ConstLRF(0.8), 
                        make_2d_input=True)
som_2 = spn.SpiceNetSom(n_neurons=som_size,
                        value_range_start=df_power['y_data'].min(),
                        value_range_end=df_power['y_data'].max(),
                        lrf_tuning_curve=spn.ConstLRF(0.8),
                        lrf_interaction_kernel=spn.ConstLRF(0.8))

correlation_matrix = spn.SpiceNetHcm(som_1, som_2, spn.ConstLRF(0.8), spn.ConstLRF(0.8))

spice_net = spn.SpiceNet(correlation_matrix)

In [25]:
input_data = df_power[['x_data', 'z_data']].apply(lambda row: row.to_numpy(), axis=1).tolist()
spice_net.fit(input_data, df_power['y_data'].tolist(), 10, 10, print_output=True)


100%|██████████| 300/300 [01:32<00:00,  3.25it/s]

Time spend on the Components: 
Som: 63.48996949195862 s | Convolution Matrix: 28.64819884300232 s


In [26]:
#spn.plot_som(som_1, True)
spn.plot_som(som_2, True)
spn.plot_hcm(correlation_matrix)

In [27]:
errors = []

for i in range(100):
    test_data = df_power.sample()
    inti_test_value = test_data[['x_data', 'z_data']].iloc[0].to_numpy()
    result_test_value = test_data['y_data'].iloc[0]
    predicted = spice_net.decode(inti_test_value)
    errors.append(abs(predicted - result_test_value) / 2 * 100)
    print(f'Error: {errors[-1]:.6f},\t Predicted: {predicted:.6f},\t Actual: {result_test_value:.6f}')

print(f'Mean Error in % {np.mean(np.array(errors))}')
print(f'Median Error in % {np.median(np.array(errors))}')

Error: 4.335529,	 Predicted: 0.582997,	 Actual: 0.669708


ValueError: math domain error